In [1]:
library(conText)
library(quanteda)
library(dplyr)
library(ggplot2)

conText v3.0.1 successfully loaded. Note this version of the package comes with significant changes to the conText() function.
                        See Quick Start Guide for help getting started and instructions for accessing older versions of the package:
 https://github.com/prodriguezsosa/conText/blob/master/vignettes/quickstart.md

Package version: 4.3.1
Unicode version: 15.0
ICU version: 73.2

Parallel computing: disabled

See https://quanteda.io for tutorials and examples.


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
all_toks_final <- readRDS("all_toks_ngram_0412.rds")

In [3]:
glove <- readRDS("glove.rds")
khodak <- readRDS("khodakA.rds")

In [4]:
dict_terms <- readRDS("Results_Files/dict_terms_final_0510.rds")

## Build Subsamples

In [15]:
indices <- seq_len(ndoc(all_toks_final))

# Sample
set.seed(7)
subsamples <- sample(1:20, size=length(indices), replace=TRUE,prob=rep(0.05, times=20))

# Extract the subsamples from the tokens object
subsampled_tokens <- split(all_toks_final, subsamples)

In [23]:
subsample_sizes <- sapply(subsampled_tokens, length)

In [24]:
write.csv(subsample_sizes, "subsample_sizes.csv")

## ALC Embedding

In [31]:
# Returns the raw dem (one row per context), preserving all docvars
build_anchor_dem <- function(toks_subsample, pattern, filter_terms = NULL) {
  
  anchor_toks <- tokens_context(
    x       = toks_subsample,
    pattern = pattern,
    window  = 6L
  )
  
  if (!is.null(filter_terms)) {
    has_noise <- sapply(anchor_toks, function(ctx) any(filter_terms %in% ctx))
    anchor_toks <- anchor_toks[!has_noise]
  }
  
  anchor_dfm <- dfm(anchor_toks)
  
  anchor_dem <- dem(
    x                = anchor_dfm,
    pre_trained      = glove,
    transform        = TRUE,
    transform_matrix = khodak,
    verbose          = FALSE
  )
  
  return(anchor_dem)   # <-- raw dem, NOT dem_group'd
}

# Grouping is now a separate, flexible step
group_dem <- function(anchor_dem, by = "Year") {
  dem_group(anchor_dem, groups = anchor_dem@docvars[[by]])
}

### Generate DEM for all words by topic

In [32]:
dem_topic <- function(tp) {
  
  topic_terms <- subset(dict_terms, topic == tp)$term
  message("Processing topic: ", tp)
  
  term_dems <- lapply(setNames(topic_terms, topic_terms), function(t) {
    
    sub_dems <- lapply(1:20, function(n) {
      dem_obj <- build_anchor_dem(
        toks_subsample = subsampled_tokens[[n]],
        pattern        = t,
        filter_terms   = NULL
      )
      dem_obj@docvars$subsample <- n
      dem_obj@docvars$text_n    <- subsample_sizes[n]
      
      list(
        matrix  = as.matrix(dem_obj),
        docvars = dem_obj@docvars
      )
    })
    
    list(
      matrix  = do.call(rbind, lapply(sub_dems, `[[`, "matrix")),
      docvars = do.call(rbind, lapply(sub_dems, `[[`, "docvars"))
    )
  })
  
  saveRDS(term_dems, paste0("ALC_dems/topic_", tp, ".rds"))
  message("Saved topic: ", tp)
}

In [ ]:
for(tp in unique(dict_terms$topic)){
    dem_topic(tp)
}

### Generate dem for anchor words

In [14]:
build_anchor_dem_new <- function(toks_subsample, pattern, filter_terms = NULL) {
  
  anchor_toks <- tokens_context(
    x       = toks_subsample,
    pattern = pattern,
    window  = 6L
  )
  
  if (!is.null(filter_terms)) {
    has_noise <- sapply(anchor_toks, function(ctx) any(filter_terms %in% ctx))
    anchor_toks <- anchor_toks[!has_noise]
  }
  
  anchor_dfm <- dfm(anchor_toks)
  
  anchor_dem <- dem(
    x                = anchor_dfm,
    pre_trained      = glove,
    transform        = TRUE,
    transform_matrix = khodak,
    verbose          = FALSE
  )
  
  return(list(dem = anchor_dem, toks = anchor_toks))  # return both
}

In [15]:
liberal_patterns <- c("liberal", "liberals", "liberalism", "liberal-leaning")
conservative_patterns <- c("conservative", "conservatives", 
                            "conservativism", "conservative-leaning")
conservative_noise_terms <- c(
  "estimate", "estimates", "projection", "projections",
  "forecast", "forecasts", "investment", "investments",
  "investor", "investors"
)

Democrat_patterns <- c("democrat", "democrats")
Republican_patterns <- c("republican", "republicans")

anchor_specs <- list(
  list(name = "liberal",      patterns = liberal_patterns,      filter = NULL),
  list(name = "conservative", patterns = conservative_patterns, filter = conservative_noise_terms),
  list(name = "democrat",     patterns = Democrat_patterns,     filter = NULL),
  list(name = "republican",   patterns = Republican_patterns,   filter = NULL)
)

In [17]:
anchor_dems <- lapply(anchor_specs, function(spec) {
  
  sub_dems <- lapply(1:20, function(n) {
    result  <- build_anchor_dem_new(
      toks_subsample = subsampled_tokens[[n]],
      pattern        = spec$patterns,
      filter_terms   = spec$filter
    )
    dem_obj <- result$dem
    toks    <- result$toks
    
    dem_obj@docvars$subsample <- n
    dem_obj@docvars$text_n    <- subsample_sizes[n]
    
    list(
      matrix   = as.matrix(dem_obj),
      docvars  = dem_obj@docvars,
      contexts = sapply(toks, paste, collapse = " ")  # one string per context
    )
  })
  
  list(
    matrix   = do.call(rbind,  lapply(sub_dems, `[[`, "matrix")),
    docvars  = do.call(rbind,  lapply(sub_dems, `[[`, "docvars")),
    contexts = do.call(c,      lapply(sub_dems, `[[`, "contexts"))
  )
})

names(anchor_dems) <- sapply(anchor_specs, `[[`, "name")
saveRDS(anchor_dems, "ALC_dems/anchors.rds")

1762 instances of "liberal" found.
28 instances of "liberal-leaning" found.
135 instances of "liberalism" found.
501 instances of "liberals" found.
1812 instances of "liberal" found.
12 instances of "liberal-leaning" found.
180 instances of "liberalism" found.
597 instances of "liberals" found.
1832 instances of "liberal" found.
20 instances of "liberal-leaning" found.
133 instances of "liberalism" found.
585 instances of "liberals" found.
1947 instances of "liberal" found.
26 instances of "liberal-leaning" found.
108 instances of "liberalism" found.
523 instances of "liberals" found.
1830 instances of "liberal" found.
18 instances of "liberal-leaning" found.
130 instances of "liberalism" found.
612 instances of "liberals" found.
1811 instances of "liberal" found.
13 instances of "liberal-leaning" found.
112 instances of "liberalism" found.
584 instances of "liberals" found.
1809 instances of "liberal" found.
23 instances of "liberal-leaning" found.
100 instances of "liberalism" found.

## Face Validity

In [30]:
results_df <- readRDS("Results Files/results_df_label_combined.rds")

In [ ]:
results_df %>%
    filter(term == "vaccine") %>%
    ggplot(aes(x= year, y= pol_score, color = group)) +
    geom_point() +
    geom_line() +
    scale_x_continuous(
        breaks = c(1981, 1989, 1993, 2001, 2009, 2017, 2021),
        labels = c("1981\nReagan", "1989\nBush", "1993\nClinton",
                   "2001\nBush", "2009\nObama", "2017\nTrump", "2021\nBiden")) +
    scale_color_manual(values = c("#FFC20A", "#0C7BDC")) +
    theme_classic() +
    labs(x = "Year",
         y = "Politicization Score") + 
    guides(color = guide_legend(title = "")) +
    theme(plot.title = element_text(size= 20), 
          plot.subtitle=element_text(size=18), 
          strip.text = element_text(size = 15))

In [80]:
ggsave("vaccine_graph.png")

Saving 6.67 x 6.67 in image


## Close Reading

In [5]:
anchor_dems <- readRDS("ALC_dems/anchors.rds")

In [11]:
close_reading <- function(term_dem, anchor_dems, n_top = 20, 
                           filter_source = NULL, filter_year = NULL) {
  
  mat     <- term_dem$matrix
  docvars <- term_dem$docvars
  
  # Optional filtering
  keep <- rep(TRUE, nrow(mat))
  if (!is.null(filter_source)) keep <- keep & docvars$Source == filter_source
  if (!is.null(filter_year))   keep <- keep & docvars$Year   == filter_year
  
  mat     <- mat[keep, ]
  docvars <- docvars[keep, ]
  
  # Group anchors by year
  lib_by_year <- rowsum(anchor_dems$liberal$matrix, 
                         anchor_dems$liberal$docvars$Year) / 
                 as.numeric(table(anchor_dems$liberal$docvars$Year))
  
  con_by_year <- rowsum(anchor_dems$conservative$matrix,
                         anchor_dems$conservative$docvars$Year) /
                 as.numeric(table(anchor_dems$conservative$docvars$Year))
  
  # Cosine of each context against its own year's anchor
  cos_scores <- sapply(seq_len(nrow(mat)), function(i) {
    yr <- as.character(docvars$Year[i])
    if (!yr %in% rownames(lib_by_year) | !yr %in% rownames(con_by_year)) return(NA)
    cos_lib <- lsa::cosine(mat[i, ], lib_by_year[yr, ])
    cos_con <- lsa::cosine(mat[i, ], con_by_year[yr, ])
    (cos_lib + cos_con) / 2
  })
  
  # Find contexts closest to the median score
  med         <- max(cos_scores, na.rm = TRUE)
  dist_to_med <- abs(cos_scores - med)
  median_idx  <- order(dist_to_med)[1:n_top]
  
  data.frame(
    Article_ID = docvars$Article_ID[median_idx],
    Year       = docvars$Year[median_idx],
    Source     = docvars$Source[median_idx],
    Type       = docvars$Type[median_idx],
    subsample  = docvars$subsample[median_idx],
    pol_score  = cos_scores[median_idx]
  )
}

In [18]:
term_dems <- readRDS("ALC_dems/topic_33.rds")

In [41]:
# All sources
closereading_results <- close_reading(term_dems[["comedian"]], anchor_dems, n_top = 20, filter_year = 2024)

closereading_results

Article_ID,Year,Source,Type,subsample,pol_score
<chr>,<dbl>,<chr>,<chr>,<int>,<dbl>
3126256091,2024,New York Times,Commentary,17,0.4439884
3068673657,2024,New York Times,News,17,0.4334860
3121375493,2024,The Washington Post,News,19,0.4101723
2908033096,2024,New York Times,Obituary,1,0.3695093
3126439199,2024,The Washington Post,News,8,0.3621006
3123193172,2024,New York Times,News,10,0.3410391
3133864220,2024,New York Times,Review,1,0.3339738
3134589402,2024,New York Times,Obituary,2,0.3210975
3147820368,2024,New York Times,News,8,0.3198518


In [20]:
get_contexts <- function(article_ids, subsample_n, pattern, anchor_dems) {
  
  toks_sub <- subsampled_tokens[[subsample_n]]
  
  ctx <- tokens_context(toks_sub, pattern = pattern, window = 6L)
  
  ctx_docvars <- docvars(ctx)
  match_idx   <- ctx_docvars$Article_ID %in% article_ids
  
  ctx_matched  <- ctx[match_idx, ]
  dvars_matched <- ctx_docvars[match_idx, ]
  
  # Build DEM for matched contexts
  ctx_dfm <- dfm(ctx_matched)
  ctx_dem <- dem(
    x                = ctx_dfm,
    pre_trained      = glove,
    transform        = TRUE,
    transform_matrix = khodak,
    verbose          = FALSE
  )
  
  # Group anchors by year
  lib_by_year <- rowsum(anchor_dems$liberal$matrix,
                         anchor_dems$liberal$docvars$Year) /
                 as.numeric(table(anchor_dems$liberal$docvars$Year))
  
  con_by_year <- rowsum(anchor_dems$conservative$matrix,
                         anchor_dems$conservative$docvars$Year) /
                 as.numeric(table(anchor_dems$conservative$docvars$Year))
  
  # Cosine for each individual context
  cos_scores <- sapply(seq_len(nrow(ctx_dem)), function(i) {
    yr <- as.character(dvars_matched$Year[i])
    if (!yr %in% rownames(lib_by_year) | !yr %in% rownames(con_by_year)) return(NA)
    cos_lib <- lsa::cosine(as.numeric(ctx_dem[i, ]), lib_by_year[yr, ])
    cos_con <- lsa::cosine(as.numeric(ctx_dem[i, ]), con_by_year[yr, ])
    (cos_lib + cos_con) / 2
  })
  
  df <- data.frame(
    context    = sapply(ctx_matched, paste, collapse = " "),
    dvars_matched
  ) %>%
    group_by(Article_ID) %>%
    mutate(occurrence = row_number()) %>%
    ungroup() %>%
    mutate(pol_score = cos_scores) %>%
    arrange(Article_ID, desc(pol_score))
  
  df
}

In [43]:
get_contexts(closereading_results$Article_ID[[1]], 
             closereading_results$subsample[[1]], 
             "comedian",
             anchor_dems = anchor_dems)

408 instances of "comedian" found.


context,pattern,Article_ID,Title,Date,Abstract,Source,Location,People,Organization,Type,Desk,Year,occurrence,pol_score
<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>
podcasters like joe rogan a former with amorphous largely libertarian political views,comedian,3126256091,"For Harris, Star Endorsements Didn't Shine as They Used To",2024-11-09,NA,New York Times,United States--US,"Swift, Taylor; Rogan, Joe; Voight, Jon; Trump, Donald J; Harris, Kamala",,Commentary,National Desk,2024,1,0.4439884
